# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khadeja-qureshi/Machine-Learning-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN and HF_TOKEN.startswith("hf_"):
    print("HF_TOKEN loaded successfully.")
else:
    print("HF_TOKEN is missing or invalid.")

HF_TOKEN loaded successfully.


In [5]:
%pip install -q duckdb pandas scikit-learn

import duckdb
import pandas as pd
import numpy as np

# Connect DuckDB to the gated Hugging Face dataset.
con = duckdb.connect()

# The token is used only in memory. It is never printed.
safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE SECRET hf_access (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_TABLE = f"""
    read_parquet(
        '{REL}/fact_content_daily_performance/month=2026-03/*.parquet'
    )
"""

print("Connected to the March 2026 warehouse partition.")

Connected to the March 2026 warehouse partition.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1. What one row means**

One row in my final feature frame represents one pseudonymized webpage belonging to one pseudonymized client. The source table is more detailed: one source row represents one webpage on one report date.

**2. Table used**

I use the `fact_content_daily_performance` table, specifically its March 2026 partition.

**3. Time window**

The feature window is March 1–15, 2026. The decision moment is the beginning of March 16. The outcome window is March 16–31, 2026.

**4. Prediction and output**

I predict whether a page's average observed daily impressions fall by more than 20% in the outcome window compared with the feature window. The eventual output will be a ranked queue of pages for content review.

**5. Deliberate exclusion**

I exclude the June 2026 sample from development because it is the final month and should remain a sealed test period. I also exclude GA4 measurements from my first five model features because GA4 availability differs across clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The honest model will use exactly five features:

1. `impressions_first15`
2. `clicks_first15`
3. `ctr_pct_first15`
4. `avg_position_first15`
5. `active_days_first15`

These are calculated only from March 1–15, before the decision moment.

### Label or proxy

`is_declining` is the proxy label. It equals 1 when average observed daily impressions in March 16–31 are more than 20% lower than in March 1–15. Otherwise, it equals 0.

This is a decline proxy. It does not prove that refreshing a page would improve its future performance.

### Context

- `client_hash_id`: used for grouping and client-level train/test splitting.
- `content_hash_id`: identifies the pseudonymized webpage.
- `report_date`: separates the feature and outcome windows.
- `ga4_data_available`: used to verify data availability.

These context fields will not be model features.

### Excluded

- June 2026: excluded from development because it is the final sealed month.
- March 16–31 performance values: excluded from honest model features because they occur after the decision moment.
- GA4 metrics: excluded from this first feature frame because availability is uneven.
- Identifiers: excluded from model features because IDs are for grouping, not learning.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# VERIFICATION QUERY 1 OF 3 — GRAIN
# One source row should represent one:
# report_date × client_hash_id × content_hash_id

grain_check = con.sql(f"""
    SELECT COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS rows_per_key
        FROM {MARCH_TABLE}
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) > 1
    )
""").df()

display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_key_groups
0,0


### Grain finding

The grain check found **0 duplicate key groups**. A result of zero supports the claim that one March source row represents one report date, one client and one content page.

In [7]:
# VERIFICATION QUERY 2 OF 3 — ROW COUNT AND DATE SPAN

slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH_TABLE}
""").df()

display(slice_summary)

,row_count,client_count,content_count,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


### Slice-size finding

The March slice contains **9841378 daily rows**, **55 clients** and **331437 content items**. Its observed date span is **2026-03-01 to 2026-03-31**.

In [8]:
# VERIFICATION QUERY 3 OF 3 — GA4 AVAILABILITY
# The assignment specifically requires an IS TRUE filter.

availability_check = con.sql(f"""
    WITH all_march_rows AS (
        SELECT *
        FROM {MARCH_TABLE}
    ),
    available_rows AS (
        SELECT *
        FROM all_march_rows
        WHERE ga4_data_available IS TRUE
    )
    SELECT
        (SELECT COUNT(*) FROM all_march_rows) AS all_rows,
        (SELECT COUNT(*) FROM available_rows) AS ga4_available_rows,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM available_rows)
            / NULLIF((SELECT COUNT(*) FROM all_march_rows), 0),
            2
        ) AS ga4_available_pct
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,ga4_available_rows,ga4_available_pct
0,9841378,413966,4.21


### Availability finding

Of **9841378** March rows, **413966** survive the filter `ga4_data_available IS TRUE`. This is **4.21%** of the March slice.

This shows that GA4 availability must be checked explicitly. A zero in an unavailable period should not automatically be interpreted as genuine zero engagement.

In [13]:
# FEATURE-BUILDING QUERY
# This is not one of the three verification queries.

feature_frame = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            -- Coverage checks
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS feature_days_present,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-16'
                                      AND DATE '2026-03-31'
            ) AS outcome_days_present,

            -- Honest feature-window information
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS impressions_first15,

            SUM(gsc_clicks) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
            ) AS clicks_first15,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                     AND gsc_impressions > 0
                    THEN gsc_avg_position * gsc_impressions
                    ELSE 0
                END
            ) / NULLIF(
                SUM(gsc_impressions) FILTER (
                    WHERE report_date BETWEEN DATE '2026-03-01'
                                          AND DATE '2026-03-15'
                ),
                0
            ) AS avg_position_first15,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-15'
                  AND gsc_impressions > 0
            ) AS active_days_first15,

            -- Outcome-window information: used only to create the label
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-16'
                                      AND DATE '2026-03-31'
            ) AS impressions_outcome

        FROM {MARCH_TABLE}
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    eligible_pages AS (
        SELECT
            *,
            impressions_first15
                / NULLIF(feature_days_present, 0)
                AS avg_daily_impressions_first15,

            impressions_outcome
                / NULLIF(outcome_days_present, 0)
                AS avg_daily_impressions_outcome
        FROM page_windows
        WHERE feature_days_present >= 10
          AND outcome_days_present >= 10
          AND impressions_first15 >= 100
    )

    SELECT
        -- Context
        client_hash_id,
        content_hash_id,

        -- Exactly five honest features
        impressions_first15,
        clicks_first15,

        100.0 * clicks_first15
            / NULLIF(impressions_first15, 0)
            AS ctr_pct_first15,

        avg_position_first15,
        active_days_first15,

        -- Label sources: never honest features
        avg_daily_impressions_first15,
        avg_daily_impressions_outcome,

        CASE
            WHEN avg_daily_impressions_outcome
                 < 0.80 * avg_daily_impressions_first15
            THEN 1
            ELSE 0
        END AS is_declining

    FROM eligible_pages
""").df()

print("Feature-frame rows:", f"{len(feature_frame):,}")
print(
    "Observed declining-label rate:",
    round(feature_frame["is_declining"].mean(), 3)
)

display(
    feature_frame[
        [
            "impressions_first15",
            "clicks_first15",
            "ctr_pct_first15",
            "avg_position_first15",
            "active_days_first15",
            "is_declining",
        ]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 77,446
Observed declining-label rate: 0.328


,impressions_first15,clicks_first15,ctr_pct_first15,avg_position_first15,active_days_first15,is_declining
0,199.0,2.0,1.005025,4.020101,13,0
1,467.0,1.0,0.214133,4.449679,13,1
2,771.0,1.0,0.129702,1.942931,14,0
3,233.0,0.0,0.000000,5.004292,13,0
4,230.0,2.0,0.869565,3.400000,13,0


### Five features and their availability

1. **`impressions_first15`** — knowable at the decision moment because it uses only impressions recorded from March 1–15.

2. **`clicks_first15`** — knowable at the decision moment because it uses only clicks recorded from March 1–15.

3. **`ctr_pct_first15`** — knowable at the decision moment because it is calculated only from March 1–15 clicks and impressions.

4. **`avg_position_first15`** — knowable at the decision moment because it uses only search-position observations recorded before March 16.

5. **`active_days_first15`** — knowable at the decision moment because it counts only days with impressions during March 1–15.

`avg_daily_impressions_outcome` and `is_declining` are label information. They must not be used as honest model features.

In [10]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score

HONEST_FEATURES = [
    "impressions_first15",
    "clicks_first15",
    "ctr_pct_first15",
    "avg_position_first15",
    "active_days_first15",
]

X_honest = (
    feature_frame[HONEST_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = feature_frame["is_declining"]
groups = feature_frame["client_hash_id"]

print("Number of pages:", len(feature_frame))
print("Number of clients:", groups.nunique())
print("Label counts:")
print(y.value_counts())

assert y.nunique() == 2, "The label needs both declining and non-declining pages."
assert groups.nunique() >= 2, "At least two clients are needed."

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_index, test_index = next(
    splitter.split(X_honest, y, groups=groups)
)

honest_model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42
)

honest_model.fit(
    X_honest.iloc[train_index],
    y.iloc[train_index]
)

honest_predictions = honest_model.predict(
    X_honest.iloc[test_index]
)

honest_score = balanced_accuracy_score(
    y.iloc[test_index],
    honest_predictions
)

print("Honest balanced accuracy:", round(honest_score, 3))

Number of pages: 77446
Number of clients: 37
Label counts:
is_declining
0    52038
1    25408
Name: count, dtype: int64
Honest balanced accuracy: 0.568


In [11]:
# DELIBERATE LEAKAGE EXPERIMENT
# This future ratio would not exist at the March 16 decision moment.

leak_experiment = feature_frame.copy()

leak_experiment["leak_future_ratio"] = (
    leak_experiment["avg_daily_impressions_outcome"]
    / leak_experiment["avg_daily_impressions_first15"]
)

LEAKY_FEATURES = HONEST_FEATURES + ["leak_future_ratio"]

X_leaky = (
    leak_experiment[LEAKY_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42
)

leaky_model.fit(
    X_leaky.iloc[train_index],
    y.iloc[train_index]
)

leaky_predictions = leaky_model.predict(
    X_leaky.iloc[test_index]
)

leaky_score = balanced_accuracy_score(
    y.iloc[test_index],
    leaky_predictions
)

print("Honest balanced accuracy:", round(honest_score, 3))
print("Leaky balanced accuracy: ", round(leaky_score, 3))

Honest balanced accuracy: 0.568
Leaky balanced accuracy:  1.0


In [12]:
leak_experiment.drop(
    columns=["leak_future_ratio"],
    inplace=True
)

assert "leak_future_ratio" not in leak_experiment.columns
assert "leak_future_ratio" not in HONEST_FEATURES

print("Leak removed successfully.")
print("Valid score kept:", round(honest_score, 3))
print("Final honest features:", HONEST_FEATURES)

Leak removed successfully.
Valid score kept: 0.568
Final honest features: ['impressions_first15', 'clicks_first15', 'ctr_pct_first15', 'avg_position_first15', 'active_days_first15']


### Leakage lesson

The honest model achieved a balanced accuracy of **0.568** using only information available before the March 16 decision moment.

After I deliberately added `leak_future_ratio`, the balanced accuracy increased to **1.000**. This feature uses average impressions from the March 16–31 outcome window and almost directly reveals whether the page crossed the 20% decline threshold used to define `is_declining`.

The perfect leaky score is not evidence of a better predictive model. It occurred because the model was given future information connected directly to the label. I removed `leak_future_ratio` and retained the honest score of **0.568** and the original five-feature list.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named limitation: unbalanced client history and uneven availability

Clients do not all have the same amount of historical data. Some clients begin contributing GSC or GA4 information later than others. This means that missing history is not necessarily random.

I reduced this problem by:

- using one mid-panel month;
- requiring at least 10 observed rows in both windows;
- checking `ga4_data_available IS TRUE`;
- excluding GA4 measurements from the first model;
- keeping June 2026 sealed for later testing.

However, the resulting feature frame may represent pages with stronger data coverage more heavily than pages with shorter histories.

The label is also only a short-window decline proxy. It can identify observed changes in impressions, but it cannot prove that refreshing a page causes traffic or ranking improvements.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.